In [1]:
# Setup
import argparse
import os, sys
import numpy as np
import pandas as pd
import time
import gc
import re
import random
import copy
import math
from matplotlib import pyplot as plt
import io
import json
import pickle
import time

import transformers
import torch
import openai

from transformers import LlamaForCausalLM, LlamaTokenizerFast

# Evaluate openai with the current paraphrasing strategy.
# Batch up 20 paras per Q, see ASR.

KEYPATH = ""
with open(KEYPATH, 'r') as f:
    API_KEY = f.readline()
    API_KEY = API_KEY.rstrip('\n')

OPENAI_KEYPATH = "/home/jlim/MS_Project//OpenAI_Api_free"
with open(OPENAI_KEYPATH, 'r') as f:
    OPENAI_API_KEY = f.readline()
    OPENAI_API_KEY = OPENAI_API_KEY.rstrip('\n')

# LLaMA model links:
# https://huggingface.co/meta-llama/Llama-2-7b-chat/tree/main
# Nevermind, use this similar model instead:
# https://huggingface.co/meta-llama/Llama-2-7b-hf
# Example code for using transformers with the model. I used this as a starting point:
# https://medium.com/@lucnguyen_61589/llama-2-using-huggingface-part-1-3a29fdbaa9ed

# Try a bigger model...
# TEST_MODEL = "Llama-3.1-8B-Instruct"
# MODEL_REPO = "meta-llama/Llama-3.1-8B-Instruct"  # Other model does not have a chat template; no chat support?

LOCAL_MODEL = "Llama-3.2-3B-Instruct" # This model does the paraphrasing.
MODEL_REPO = "meta-llama/Llama-3.2-3B-Instruct"  # Other model does not have a chat template; no chat support?

GEN_TEMP = 1.0

LOCAL_EOT_STR = "<|eot_id|>"

LOCAL_DEVICE_STR = 'cuda'


# Training sets
PAWS_WIKI_PATH = '/home/jlim/MS_Project/validator_validation/paws_wiki_labeled_final/final/train.tsv'
PAWS_QQP_PATH = '/home/jlim/MS_Project/validator_validation/paws_qqp/output/train.tsv'

# PAWS_WIKI_PATH = '/home/jeremy/Documents/WPI_MS/Q_Inoculate/Q_Inoculate/validator_validation/paws_wiki_labeled_final/final/train.tsv'
# PAWS_QQP_PATH = '/home/jeremy/Documents/WPI_MS/Q_Inoculate/Q_Inoculate/validator_validation/paws_qqp/output/train.tsv'


# For short answers
MAX_NEW_TOKENS = 10

# For paraphrasing.
MAX_PARAPHRASE_TOKENS = 1000  # Roughly 2x the number of tokens for the longest question.

CHOICES = ["A", "B", "C", "D"]

BATCH_SIZE = 1  # one at a time for now. Avoids padding issues.


DATA_DIR = "/home/jlim/MS_Project/MMLU_Baseline/data"

ANSWER_STRING_MAP = {'0': "A", '1': "B", '2': "C", '3': "D"}

# print("Data directory: " + str(DATA_DIR))

PARSE_RETRIES = 1 # 5 usually. 1 for no sampling.

# Function Defs
def eval_q(pipeline, q1, q2):
    # return the label that the model answers for a given prompt.
    # True means paraphrase, False means no paraphrase.

    # More testing - best so far
    verify_prompt = ("Do sentence 1 and sentence 2 have extremely different meanings? "
        "Remember that some sentences that are phrased differently have the same meaning. "
        "The sentences may appear different, but the wording may actually imply a very similar meaning. "
        "For your answer, please state either \"yes.\" or \"no.\" at the start of your answer."
        "\nSentence 1: '{q1}'"
        "\nSentence 2: '{q2}'\n")
    

    verify_msgs = [{"role": "user", "content": verify_prompt.format(q1=q1, q2=q2)}]

    successful_parse = False
    fail_count = 0
    while not successful_parse:

        prompt = pipeline.tokenizer.apply_chat_template(verify_msgs, tokenize=False, add_generation_prompt=True)

        # print("Model Prompt~~~~~~~~~~~~~~")
        # print(prompt)

        encoded_verify_request = pipeline.tokenizer(prompt, return_tensors="pt",
                                                    padding=False)  # Keeping padding off for now.

        # No gradients!
        with torch.no_grad():
            # Put on proper device?
            # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
            encoded_verify_request.to(LOCAL_DEVICE_STR)

            generated_output = pipeline.model.generate(**encoded_verify_request, do_sample=False, temperature=GEN_TEMP,
                                                    max_new_tokens=MAX_NEW_TOKENS,
                                                    return_dict_in_generate=True, output_scores=True, output_logits=True,
                                                    pad_token_id=pipeline.tokenizer.eos_token_id)

            # generated_output = generated_output.sequences[0][-MAX_NEW_TOKENS:]
            generated_output = generated_output.sequences[0][encoded_verify_request['input_ids'].shape[1]:]

            

            # Detokenize, convert to lowercase.
            generated_output = pipeline.tokenizer.decode(generated_output).lower()

            # Keep tensor memory clear!
            del encoded_verify_request

        # print("Model Response~~~~~~~~~~~~~~")
        # print(generated_output)

        if re.search("^yes", generated_output):  # Begin is "^yes"
            # yes
            # print("Parsed yes")
            # return True
            return False
        elif re.search("^no", generated_output):  # Begin is "^no"
            # no
            # print("Parsed no")
            # return False
            return True

        # if re.search(re.escape("yes.<|eot_id|>") + "$", generated_output):  # Begin is "^yes"
        #     # yes
        #     # print("Parsed yes")
        #     return True
        # elif re.search(re.escape("no.<|eot_id|>") + "$", generated_output):  # Begin is "^no"
        #     # no
        #     # print("Parsed no")
        #     return False

        else:
            fail_count += 1
            print("Response validation failure. Fail count: " + str(fail_count))
            # print("Failing Validation response:")
            # print(generated_output)
            if fail_count > PARSE_RETRIES:
                print("Could not generate a valid response!")
                return None
            
MAX_PARA_RETRIES = 20

PARA_TEMP = 1.0  # default: 1.0

# For paraphrasing.
MAX_PARAPHRASE_TOKENS = 1000  # Roughly 2x the number of tokens for the longest question.
            
def paraphrase_question(pipeline, q1, max_retries=MAX_PARA_RETRIES, temp=PARA_TEMP, max_new_toks=MAX_PARAPHRASE_TOKENS):
    # Paraphrase a question.
    # Multiple tries may be required.
    # Return (newquestion, num_retries). newquestion may be null if we exceeded the maximum number of retries.

    paraphrase_prompt = ("Please paraphrase the following question by substituting words or phrases. "
                     "The paraphrased question should have the same meaning as the original. "
                     "Your response should be surrounded in quotes like the following format: \'<new question>\'.\nOriginal Question: \'{}\'\n")
    
    char_len_proportion = 0.5  # Paraphrase must be within +/- this proportion of the length of the original. Rough metric to remove incorrect responses.

    paraphrase_msgs = [{"role": "user", "content": paraphrase_prompt.format(q1)}]  # question_no_choices

    successful = False
    parse_fail_count = 0
    para_q = None

    para_count = 0

    prompt = pipeline.tokenizer.apply_chat_template(paraphrase_msgs, tokenize=False, add_generation_prompt=True)
    encoded_paraphrase_request = pipeline.tokenizer(prompt, return_tensors="pt", padding=False) # Turning padding off for now

    # Put on proper device?
    # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
    encoded_paraphrase_request.to(LOCAL_DEVICE_STR)

    while not successful:
        para_count += 1
        print("Paraphrase attempt: " + str(para_count))
        if para_count > max_retries:
            return (None, para_count)

        generated_output = pipeline.model.generate(**encoded_paraphrase_request, do_sample=True, temperature=temp, max_new_tokens=max_new_toks,
                                            return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=pipeline.tokenizer.eos_token_id)

        # generated_output = generated_output.sequences[0][-MAX_PARAPHRASE_TOKENS:]
        generated_output = generated_output.sequences[0][encoded_paraphrase_request['input_ids'].shape[1]:]

        # Detokenize
        generated_output = pipeline.tokenizer.decode(generated_output)

        # print("Model output ~~~~~~~~~~~~~~~~~~~")
        # print(generated_output)
        # continue

        # result = re.search("\"\".*\"\"", generated_output)

        # The prompt has a string that matches this. Ignore this by getting only the last match!
        # all_matches = re.findall("\"\".*\"\"", generated_output)
        all_matches = re.findall("[\"\'].*[\"\']", generated_output)  # Switching to single double quotes
        # all_matches = all_matches[1:]  # Exclude first match; that was the original question.

        # The llm tends to put many extra quotes around stuff, chaotically. We look for both quote cases.

        # Just look at the last match.
        if len(all_matches) == 0:
            result = None
        else:
            result = all_matches[-1]

        if result is not None:
            # para_q = generated_output[result.start():result.end()]
            para_q = result
            para_q = para_q.strip("\"\'")

            # print("Paraphrase response:")
            # print(generated_output)
            # print("~~~~~~~~~~~~~~~~~~~~~")

            # print("Number of parse retries: " + str(fail_count))
            # print("Paraphrased question; original:")
            # print(para_q)
            # print(question_no_choices)

            # print("Comparing to: " + str(question_no_choices))

            is_paraphrase = eval_q(pipeline, q1, para_q)
            is_identical = (q1 == para_q)
            is_nonempty = len(para_q) > 0
            is_proportion = (len(para_q) > len(q1) - char_len_proportion*len(q1)) and (len(para_q) < len(q1) + char_len_proportion*len(q1))

            # print("Orig len: " + str(len(question_no_choices)))
            # print("Para len: " + str(len(para_q)))

            # print("Is paraphrase?: " + str(is_paraphrase))
            # print("Is identical?: " + str(is_identical))
            # print("Is nonempty?: " + str(is_nonempty))
            # print("Meets proportion requirements?: " + str(is_proportion))

            if is_paraphrase and (not is_identical) and is_nonempty and is_proportion:
                # Keep tensor memory clear!
                del encoded_paraphrase_request
                successful = True

                return (para_q, para_count)
                

        else:
            parse_fail_count += 1
            # print("Failing Paraphrase response:")
            # print(generated_output)
            if parse_fail_count > PARSE_RETRIES:
                # Fallback to not paraphrasing?
                print("Could not parse a valid paraphrase response!")
                # num_rephrases[-1] = -1
                
                return (None, para_count)

    return (None, para_count)


# Perplexity calculating functions.
def get_sequence_odds_log(logits_tensor, selected_tokens_list):
    # Compare to softmaxing & mult.

    forced_logits = logits_tensor[list(range(len(selected_tokens_list))), selected_tokens_list]

    # Softmax, but in logits world.
    forced_token_logits = forced_logits - torch.logsumexp(logits_tensor, dim=1)

    sequence_prob = torch.sum(forced_token_logits)

    # In log world, higher value means higher likelyhood still.
    return sequence_prob.item()


def get_prompt_gen_logprob(pipeline, prompt, q1, q2):

    # Trying to diagnose memory leaks:
    # https://discuss.pytorch.org/t/how-to-check-the-gpu-memory-being-used/131220
    

    # q1 is substituted into the prompt.
    model_query = [{"role": "user", "content": prompt.format(q1=q1)}]

    # Get the tokens for our desired response.
    test_response = pipeline.tokenizer(q2, padding=False, return_tensors="pt")  # Keeping padding off for now.

    test_response.to(LOCAL_DEVICE_STR)

    # other_response = pipeline.tokenizer("." + q2 +".", padding=False)

    # print(pipeline.tokenizer.decode(test_response['input_ids']))
    # print(pipeline.tokenizer.decode(other_response['input_ids']))

    test_ids = test_response['input_ids'][0][1:] # Skip the first token; usually adds begin-of-text token.

    test_num_tokens = test_ids.shape[0]

    # one-time query
    prompt = pipeline.tokenizer.apply_chat_template(model_query, tokenize=False, add_generation_prompt=True)


    encoded_request = pipeline.tokenizer(prompt, return_tensors="pt",
                                                padding=False)  # Keeping padding off for now.

    # print("Model Prompt~~~~~~~~~~~~~~")
    # print(prompt)

    logits_sequence = None

    # No gradients!
    with torch.no_grad():

        in_ids = copy.deepcopy(encoded_request)

        in_ids.to(LOCAL_DEVICE_STR)

        in_ids = in_ids['input_ids']

        # print("Device id: " + str(in_ids.get_device()))

        all_tokens = torch.concat([in_ids, torch.unsqueeze(test_ids, dim=0)], dim=1)

        # Code adapted from here: https://discuss.huggingface.co/t/generate-without-using-the-generate-method/11379
        # TODO: Can transform into batched form? We know all of the needed variations ahead of time...
        # NOTE: I think the model response isn't some starting character, but it could affect the logic here...
        for a in range(test_num_tokens):
            # NOTE: This is not memory efficient! Causes horrible issues...
            # debug_mem_use()

            # the_idx = in_ids.shape[1] + a

            # outputs = pipeline.model(input_ids=all_tokens[:,:the_idx])
            outputs = pipeline.model(input_ids=in_ids)
            logits = outputs.logits[:, -1, :]
            # logits_sequence.append(logits)  # LEAK?

            # Concat in loop, not at the end
            if logits_sequence is None:
                logits_sequence = logits
            else:
                logits_sequence = torch.concat([logits_sequence, logits], dim=0)

            # Force the next token to be what I want.
            # Put this on the GPU - TODO: Make more efficient.
            # add_tensor = torch.tensor([[test_ids[a]]])
            # add_tensor.to(DEVICE_STR)

            # print("Device id: " + str(in_ids.get_device()))
            # print("Device id: " + str(add_tensor.get_device()))

            # # new_ids = torch.concat([in_ids, torch.unsqueeze(torch.unsqueeze(test_ids[a], dim=0), dim=0)], dim=1)
            # del in_ids # Explicitly drop old reference.
            in_ids = torch.concat([in_ids, torch.unsqueeze(torch.unsqueeze(test_ids[a], dim=0), dim=0)], dim=1)
            # del outputs
            # del logits

            # Not working; clear cache?
            torch.cuda.empty_cache()

            # in_ids = torch.concat([in_ids, torch.unsqueeze(torch.unsqueeze(test_ids[a], dim=0), dim=0)], dim=1)
            # in_ids = torch.concat([in_ids, predicted_id])

        del in_ids
        del test_response

        # return 0.0

        # Compute logsumexp.
        # logits_sequence = torch.concat(logits_sequence, dim=0)

        # Re-index by our given sequence.

        # Remembering how to use logsumexp:
        # https://gregorygundersen.com/blog/2020/02/09/log-sum-exp/

        # print("Before exit:")
        # debug_mem_use()

        return get_sequence_odds_log(logits_sequence, test_ids) # , alpha=0.8



def debug_mem_use():
    # Try to debug gpu memory issues
    torch.cuda.empty_cache()
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print("torch.cuda.memory_allocated: %fGB"%(torch.cuda.memory_allocated(0)/1024/1024/1024))
    print("torch.cuda.memory_reserved: %fGB"%(torch.cuda.memory_reserved(0)/1024/1024/1024))
    print("torch.cuda.max_memory_reserved: %fGB"%(torch.cuda.max_memory_reserved(0)/1024/1024/1024))
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

# 
def paraphrase_by_perplexity(pipeline, q1, max_retries=MAX_PARA_RETRIES, temp=PARA_TEMP, max_new_toks=MAX_PARAPHRASE_TOKENS):
    # Paraphrase a question.
    # Multiple tries may be required.
    # Return (newquestion, num_retries). newquestion may be null if we exceeded the maximum number of retries.

    # paraphrase_prompt = ("Please paraphrase the following question by substituting words or phrases. "
    #                  "The paraphrased question should have the same meaning as the original. "
    #                  "Your response should be surrounded in quotes like the following format: \'<new question>\'.\nOriginal Question: \'{}\'\n")
    
    paraphrase_prompt = ("Please paraphrase the following question by substituting words or phrases. "
                    "The paraphrased question should have the same meaning as the original. "
                    "Your response should begin with the paraphrased question surrounded in quotes like the following format: \'<new question>\'.\nOriginal Question: \'{q1}\'\n")
    
    char_len_proportion = 1.0  # Paraphrase must be within +/- this proportion of the length of the original. Rough metric to remove incorrect responses.

    paraphrase_msgs = [{"role": "user", "content": paraphrase_prompt.format(q1=q1)}]  # question_no_choices

    successful = False
    parse_fail_count = 0
    para_q = None

    para_count = 0

    prompt = pipeline.tokenizer.apply_chat_template(paraphrase_msgs, tokenize=False, add_generation_prompt=True)
    encoded_paraphrase_request = pipeline.tokenizer(prompt, return_tensors="pt", padding=False) # Turning padding off for now

    print("Paraphrase prompt char length: " + str(len(prompt)))

    # Put on proper device?
    # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
    encoded_paraphrase_request.to(LOCAL_DEVICE_STR)

    compare_perplexity = get_prompt_gen_logprob(pipeline, paraphrase_prompt, q1, q1)

    while not successful:
        para_count += 1
        # print("Paraphrase attempt: " + str(para_count))

        # if para_count > 10:
        #     print("Break")

        if para_count > max_retries:
            print("Num paraphrase retries: " + str(para_count))
            return (None, para_count)

        generated_output = pipeline.model.generate(**encoded_paraphrase_request, do_sample=True, temperature=temp, max_new_tokens=max_new_toks,
                                            return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=pipeline.tokenizer.eos_token_id)

        # generated_output = generated_output.sequences[0][-MAX_PARAPHRASE_TOKENS:]
        generated_output = generated_output.sequences[0][encoded_paraphrase_request['input_ids'].shape[1]:]

        # Detokenize
        generated_output = pipeline.tokenizer.decode(generated_output)

        # print("Model output ~~~~~~~~~~~~~~~~~~~")
        # print(generated_output)
        # continue

        # result = re.search("\"\".*\"\"", generated_output)

        # The prompt has a string that matches this. Ignore this by getting only the last match!
        # all_matches = re.findall("\"\".*\"\"", generated_output)
        all_matches = re.findall("[\"\'].*[\"\']", generated_output)  # Switching to single double quotes
        # all_matches = all_matches[1:]  # Exclude first match; that was the original question.

        # The llm tends to put many extra quotes around stuff, chaotically. We look for both quote cases.

        # Just look at the last match.
        if len(all_matches) == 0:
            result = None
        else:
            result = all_matches[-1]

        if result is not None:
            # para_q = generated_output[result.start():result.end()]
            para_q = result
            para_q = para_q.strip("\"\'")

            # print("Starting question:")
            # print(q1)
            # print(".....................")
            # print("Paraphrase response:")
            # print(para_q)
            # print("~~~~~~~~~~~~~~~~~~~~~")

            # print("Number of parse retries: " + str(fail_count))
            # print("Paraphrased question; original:")
            # print(para_q)
            # print(question_no_choices)

            # print("Comparing to: " + str(question_no_choices))

            is_nonempty = len(para_q) > 0
            if is_nonempty:

                # 
                gen_perplexity =  get_prompt_gen_logprob(pipeline, paraphrase_prompt, q1, para_q)


                is_paraphrase = ((gen_perplexity - compare_perplexity) >= 0.0)  # Use perplexity as opposed to a separate eval.
                # print("Perplexity difference: " + str(gen_perplexity - compare_perplexity))

                # More likely to be true?
                is_identical = (q1 == para_q)
                
                is_proportion = (len(para_q) > len(q1) - char_len_proportion*len(q1)) and (len(para_q) < len(q1) + char_len_proportion*len(q1))

                # print("Orig len: " + str(len(q1)))
                # print("Para len: " + str(len(para_q)))

                # print("Perplexity okay?: " + str(is_paraphrase))
                # print("Is identical?: " + str(is_identical))
                # print("Is nonempty?: " + str(is_nonempty))
                # print("Meets proportion requirements?: " + str(is_proportion))


                # Ignore proportion!  and is_proportion
                if is_paraphrase and (not is_identical):
                    # Keep tensor memory clear!
                    del encoded_paraphrase_request
                    successful = True

                    print("Num paraphrase retries: " + str(para_count))
                    return (para_q, para_count)
                

        else:
            parse_fail_count += 1
            # print("Failing Paraphrase response:")
            # print(generated_output)
            if parse_fail_count > PARSE_RETRIES:
                # Fallback to not paraphrasing?
                print("Could not parse a valid paraphrase response!")
                # num_rephrases[-1] = -1
                
                print("Num paraphrase retries: " + str(para_count))
                return (None, para_count)

    print("Num paraphrase retries: " + str(para_count))
    return (None, para_count)


POLL_PERIOD = 5

RESPONSE_RETRIES = 5

def bulk_eval(chat_completion_param_set: list, openai_client):
    # Use the typical api; batch completion is waaaay to slow for testing.
    # https://github.com/openai/openai-python

    response_strs = []
    # Not elegant, but faster.


    for idx, params in enumerate(chat_completion_param_set):
        response_str = ''
        response_tries = 0
        print("Completion: " + str(idx+1) + "/" + str(len(chat_completion_param_set)))

        while(len(response_str) == 0):  # Having trouble with empty responses.
            response_tries += 1
            if(response_tries > RESPONSE_RETRIES) :
                print("Couldn't get response...")
                break

            response = openai_client.chat.completions.create(**params)
            response_str = response.choices[0].message.content


        # print("Response: " + response_str)
        response_strs.append(response_str)

    return response_strs

def batch_eval(chat_completion_param_set: list, openai_client):
    # Use openai's batch api; send a large number of chat completion requests, then block for a response.

    batch_file = io.BytesIO()

    # num_completions = len(chat_completion_param_set)

    # make an in-memory json file
    for idx, params in enumerate(chat_completion_param_set):
        # Add custom id, because return order is not guaranteed
        request_obj = {
            "custom_id": str(idx),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": params,
        }

        # Assuming utf-8????
        dumpstr = json.dumps(request_obj).encode()
        # json.dump(params, batch_file)
        batch_file.write(dumpstr)

        batch_file.write("\n".encode())  # jsonl format

    batch_file.seek(0)
    # print("File contents: ")
    # print(batch_file.read())

    # Reference: https://platform.openai.com/docs/api-reference/files?lang=python
    # https://platform.openai.com/docs/guides/batch

    # 0: List current files.
    # files_list = openai_client.files.list()
    #
    # # When we need to clear files
    # for f in files_list:
    #     delete_stat = openai_client.files.delete(f.id)
    # files_list = openai_client.files.list()

    # 1: Upload batch file
    uploaded_file = openai_client.files.create(file=batch_file, purpose="batch")
    # Debugging
    # uploaded_file = openai_client.files.create(None)

    fid = uploaded_file.id

    # 2: Tell OpenAI to run the particular batch file
    # NOTE: Correct endpoint?
    batch = openai_client.batches.create(input_file_id=fid,
                                         endpoint="/v1/chat/completions", # /v1/completions
                                         completion_window='24h')
    batch_id = batch.id

    # JL - having weird issues with batches not starting. So adding a simple retry loop for stability?
    # For whatever reason
    started = False
    retry_count = 0
    while not started:
        time.sleep(POLL_PERIOD)
        batch_update = openai_client.batches.retrieve(batch_id)

        if batch_update.status != 'in_progress':
            if batch_update.status == 'failed':
                if type(batch_update.errors.data[0]) == openai.types.batch_error.BatchError:
                    retry_count += 1
                    print("Retrying, count: " + str(retry_count))

                    batch = openai_client.batches.create(input_file_id=fid,
                                                         endpoint="/v1/chat/completions",  # /v1/completions
                                                         completion_window='24h')
                    batch_id = batch.id
                else:
                    raise Exception("Batch ended with status: " + batch_update.status)
        else:
            started = True

    print("Batch started successfully")
    # 3: Wait for results (can poll periodically I guess)
    output_file_id = None
    finished = False
    while not finished:
        time.sleep(POLL_PERIOD)  # Wait period, so as to not spam polling...
        batch_update = openai_client.batches.retrieve(batch_id)

        # check for all terminal states
        status = batch_update.status  # error, 'method' parameter...
        print("Batch status: " + str(status))
        if status == 'failed' or status == 'expired' or status == 'cancelling' or status == 'cancelled':
            # delete_status = openai_client.files.delete(fid)  # Clean up failed batch file.
            raise Exception("Batch ended with status: " + status)
        elif status == 'completed':
            finished = True
            output_file_id = batch_update.output_file_id

    # Wait a little
    time.sleep(POLL_PERIOD)

    # 4: When results are ready, pull them down.
    response_file = openai_client.files.content(output_file_id)

    # file format notes:
    # look at response_file.text in example.jsonl, to format parsing the response
    # Parse response
    response_lines = response_file.text.split("\n")
    responses = [json.loads(x) for x in response_lines if len(x) > 0]

    # 5: Clean up files created on openai's api.
    delete_status = openai_client.files.delete(fid)
    # 5.5: Delete output file too.
    delete_status2 = openai_client.files.delete(output_file_id)

    # 6: Use custom ids to re-order and return responses
    responses = sorted(responses, key=lambda x: x["custom_id"])

    response_strs = [x["response"]["body"]["choices"][0]["message"]["content"] for x in responses]

    # list of top logprobs: x["response"]["body"]["choices"][0]["logprobs"]["content"][0]["top_logprobs"]

    # top_tokens_lists = [[y["token"] for y in x["response"]["body"]["choices"][0]["logprobs"]["content"][0]["top_logprobs"]] for x in responses]
    # top_logprobs_lists = [[y["logprob"] for y in x["response"]["body"]["choices"][0]["logprobs"]["content"][0]["top_logprobs"]] for x in responses]

    # print("done")
    # files_list = openai_client.files.list()
    # print("Files remaining on openAI api: " + str(len(files_list.data)))

    return response_strs


# RESPONSE_MAX_NEW_TOKENS = 50 
RESPONSE_MAX_NEW_TOKENS = 25 # For performance

# RESPONSE_MAX_NEW_TOKENS = 240  
# RESPONSE_MAX_NEW_TOKENS = 300  # Not sure how long this should be? Updating to 300 (originally 240), due to some responses hanging a bit.

# These models might need a large number of tokens...
# RESPONSE_MAX_NEW_TOKENS = 1000

ANSWER_TEMP = 1

def build_question_batch(model_identifier, question_list, max_new_tokens=RESPONSE_MAX_NEW_TOKENS):

    # prompt_string = ("Given the following question and four candidate answers (A, B, C and D), choose the best answer.\nQuestion: {}\n"
    #                     "Your response should end with \"The best answer is [the_answer_letter]\" where the [the_answer_letter] is one of A, B, C or D.")
    
    # Prompt for better performance, compliance on chatGPT:
    # prompt_string = ("Given the following question and four candidate answers (A, B, C and D), choose the best answer.\nQuestion: {}\n"
    #                 "Your response should begin with \"The best answer is [the_answer_letter]\" where the [the_answer_letter] is one of A, B, C or D.")
    
    prompt_string = ("Given the following question and four candidate answers (A, B, C and D), choose the best answer.\nQuestion: {}\n"
                "Your response should begin with \"The best answer is [the_answer_letter]\" where the [the_answer_letter] is one of A, B, C or D.\n"
                "If your response does not follow the described format, it will be incorrect.")

    batch_param_sets = []

    for q in question_list:

        # N
        # msgs = [{"role": "system", "content": train_prompt},
        #                 {"role": "user", "content": prompt_end}]
        # No system role; we don't have few-shot examples here.
        msgs = [{"role": "user", "content": prompt_string.format(q)}]

        param_set = {
            'model': model_identifier,
            'messages': msgs,
            'logprobs': False,  # False for now.
            # 'max_tokens': RESPONSE_MAX_NEW_TOKENS, # GPT turbo
            'temperature': ANSWER_TEMP,
            'max_completion_tokens': max_new_tokens, # o3 mini
            # 'top_logprobs': 20
        }

        batch_param_sets.append(param_set)

    return batch_param_sets


def format_para_question_single(base_q, df, idx, include_answer=True):
    prompt = base_q
    k = df.shape[1] - 2  # answer index.
    for j in range(k):
        prompt += "\n{}. {}".format(CHOICES[j], df.iloc[idx, j+1])

    if include_answer:
        prompt += "\nAnswer:"
        prompt += " {}\n\n".format(df.iloc[idx, k + 1])
    return prompt

print("Setup done")

# Data setup
ATTACK_TRIES = 5  # Try this many attacks on a question at once.

UNIQUENESS_RETRIES = 5  # Try up to additional this many times to get the desired number of unique paraphrases.

# Old model first
# OPENAI_TEST_MODEL = "gpt-3.5-turbo-0125"

# Probably the best comparison
# OPENAI_TEST_MODEL = "o3-mini-2025-01-31"

# Moderate comparision
OPENAI_TEST_MODEL = "gpt-4o-mini-2024-07-18"

ANSWER_PREFIX = "The best answer is {}"

ANSWER_REG = "The best answer is [ABCD]"


2025-04-07 19:32:26.778972: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-07 19:32:26.792987: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-07 19:32:26.816278: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-07 19:32:26.816317: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-07 19:32:26.830725: I tensorflow/core/platform/cpu_feature_gua

Setup done


In [12]:


# Load MMLU dev set
# read all questions
mmlu_subjects = sorted(
    [f.split("_test.csv")[0] for f in os.listdir(os.path.join(DATA_DIR, "test")) if "_test.csv" in f])

dev_df = None
for subject in mmlu_subjects:
    # TODO: Separate by subject?
    if dev_df is None:
        dev_df = pd.read_csv(os.path.join(DATA_DIR, "dev", subject + "_dev.csv"), header=None)
    else:
        dev_df = pd.concat([dev_df, pd.read_csv(os.path.join(DATA_DIR, "dev", subject + "_dev.csv"), header=None)])
        
num_qs = dev_df.shape[0]
print("Dev dataset size: " + str(num_qs))

pipeline = None
print("Initializing model & pipeline...")
# Model, tokenizer, pipeline init
device_map_setting = LOCAL_DEVICE_STR  # 'auto'  # 'cuda'

model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    token=API_KEY,
    device_map=device_map_setting
)

# Set to eval mode!
# https://discuss.huggingface.co/t/inference-without-gradient-computation/14449
model.eval()

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_REPO, token=API_KEY) # , padding_side='left'

if torch.cuda.is_available():
    print("GPU available...")
else:
    print("No GPU available...")

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,  # Probably won't work on cpu...
    tokenizer=tokenizer,
    device_map=device_map_setting,
)

Dev dataset size: 285
Initializing model & pipeline...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda


GPU available...


In [23]:
import datetime
import itertools
# Inspiration: https://stackoverflow.com/questions/8784813/lstrip-rstrip-for-lists

def strip_eot(id_list, prompt_len, eot_token_id):
    return list(itertools.filterfalse(lambda x: x == eot_token_id, id_list[prompt_len:]))

def strip_eot_start(id_list, eot_token_id):
    start_idx = 0
    for idx in range(len(id_list)):
        if id_list[idx] != eot_token_id:
            start_idx = idx
            break

    return id_list[start_idx:]

def eval_q_batch(pipeline, orig_q, para_list):
    verify_prompt = ("Do sentence 1 and sentence 2 have extremely different meanings? "
        "Remember that some sentences that are phrased differently have the same meaning. "
        "The sentences may appear different, but the wording may actually imply a very similar meaning. "
        "For your answer, please state either \"yes.\" or \"no.\" at the start of your answer."
        "\nSentence 1: '{q1}'"
        "\nSentence 2: '{q2}'\n")
    
    prompt_lengths = []
    promptlist = []
    for test_para in para_list:
        # Get tokenized length.
        verify_msgs = [{"role": "user", "content": verify_prompt.format(q1=orig_q, q2=test_para)}]

        prompt = pipeline.tokenizer.apply_chat_template(verify_msgs, tokenize=False, add_generation_prompt=True)
        encoded_p = pipeline.tokenizer(prompt, padding=False)
        prompt_lengths.append(len(encoded_p['input_ids']))
        promptlist.append(prompt)

    # print("Model Prompt~~~~~~~~~~~~~~")
    # print(prompt)

    pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
    tokenizer.padding_side = 'left'
    batched_verify_request = pipeline.tokenizer(promptlist, return_tensors="pt",
                                                padding=True)  # Keeping padding off for now.

    # No gradients!
    with torch.no_grad():
        # Put on proper device?
        # Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
        batched_verify_request.to(LOCAL_DEVICE_STR)

        generated_output = pipeline.model.generate(**batched_verify_request, do_sample=False, temperature=GEN_TEMP,
                                                max_new_tokens=10,
                                                return_dict_in_generate=True, output_scores=True, output_logits=True,
                                                pad_token_id=pipeline.tokenizer.eos_token_id)
        
        generated_output = generated_output.sequences

        result_list = []
        for idx in range(generated_output.shape[0]):
            out_ids = generated_output[idx].cpu().detach().numpy().tolist()
            out_ids = strip_eot_start(out_ids, pipeline.tokenizer.eos_token_id)
            out_ids = out_ids[prompt_lengths[idx]:]

            single_response = pipeline.tokenizer.decode(out_ids)
            single_response = single_response.lower()

            # print("idx: " + str(idx) + " " + single_response)

            if re.search("^yes", single_response):  # Begin is "^yes"
                # yes
                # print("Parsed yes")
                # return True
                result_list.append(False)
            elif re.search("^no", single_response):  # Begin is "^no"
                # no
                # print("Parsed no")
                # return False
                result_list.append(True)
            else:
                # print("Could not generate a valid response!")
                result_list.append(None)
    
    return result_list


test_q = 'The brown fox jumped over the lazy dog.'

test_para_yes = 'The brown fox jumped over the sleepy dog.'
test_para_no = 'Well did you not see the sky? Well did you not see the sky? Well did you not see the sky? Well did you not see the sky? Well did you not see the sky?'
# test_para_no = 'Well did you not see the sky?'

test_list = [test_para_yes]*10 + [test_para_no]*10

results = eval_q_batch(pipeline, test_q, test_list)
print("Results: ")
print(str(results))


Results: 
[True, True, True, True, True, True, True, True, True, True, False, False, False, False, False, False, False, False, False, False]


In [ ]:
import datetime
import itertools
# Inspiration: https://stackoverflow.com/questions/8784813/lstrip-rstrip-for-lists

def strip_eot(id_list, prompt_len, eot_token_id):
    return list(itertools.filterfalse(lambda x: x == eot_token_id, id_list[prompt_len:]))


# Doing brief experiment to verify that batching in huggingface is useful (actually has some speedup!)

test_q = 'The brown fox jumped over the lazy dog.'

# Select random mmlu dev q...

test_count = 30

print("Start")

# Adjust tokenizer to support padding
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token

max_new_toks = 1000

# Testing generating paraphrases in batches or just via looping...
paraphrase_prompt = ("Please paraphrase the following question by substituting words or phrases. "
                    "The paraphrased question should have the same meaning as the original. "
                    "Your response should be surrounded in quotes like the following format: \'<new question>\'.\nOriginal Question: \'{}\'\n")

paraphrase_prompt2 = ("Please negate the following question by substituting the word cactus whenever possible. "
                    "Your response should be surrounded in quotes like the following format: \'<new question>\'.\nOriginal Question: \'{}\'\n")


paraphrase_msgs1 = [{"role": "user", "content": paraphrase_prompt.format(test_q)}]  # question_no_choices
paraphrase_msgs2 = [{"role": "user", "content": paraphrase_prompt2.format(test_q)}]  # question_no_choices
promptlist = []
plens = []
for a in range(test_count):

    # Return the completed template.
    if a < test_count/2:
        prompt = pipeline.tokenizer.apply_chat_template(paraphrase_msgs1, tokenize=False, add_generation_prompt=True)
        
    else:
        prompt = pipeline.tokenizer.apply_chat_template(paraphrase_msgs2, tokenize=False, add_generation_prompt=True)

    thelen = pipeline.tokenizer(prompt, return_tensors="pt", padding=False)['input_ids'].shape[1]
    plens.append(thelen)
    promptlist.append(prompt)

print("Starting sequential...")
# Individual test
output_list = []
start = datetime.datetime.now()
for p in promptlist:
    encoded_paraphrase_request = pipeline.tokenizer(p, return_tensors="pt", padding=False) # Turning padding off for now
    encoded_paraphrase_request.to(LOCAL_DEVICE_STR)

    generated_output = pipeline.model.generate(**encoded_paraphrase_request, do_sample=True, temperature=1.0, max_new_tokens=max_new_toks,
                                    return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=pipeline.tokenizer.eos_token_id)

    # generated_output = generated_output.sequences[0][-MAX_PARAPHRASE_TOKENS:]
    generated_output = generated_output.sequences[0][encoded_paraphrase_request['input_ids'].shape[1]:]

    output_list.append(generated_output)

    # generated_output = pipeline.tokenizer.decode(generated_output)
    # print("Output length: " + str(len(generated_output)))
elapsed = datetime.datetime.now() - start

print("Total time to do " + str(test_count) + " paraphrases sequentially: " + str(elapsed.total_seconds()))

print("Sequential results: ")
for resp in output_list:
    print(pipeline.tokenizer.decode(resp))
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')

print("Starting batch...")

start = datetime.datetime.now()
# Batched test
batched_paraphrase_request = pipeline.tokenizer(promptlist, return_tensors="pt", padding=True) # Turning padding off for now

# Put on proper device?
# Inspiration: https://discuss.huggingface.co/t/device-map-auto-with-error-expected-all-tensors-to-be-on-the-same-device/31938/6
batched_paraphrase_request.to(LOCAL_DEVICE_STR)


generated_output = pipeline.model.generate(**batched_paraphrase_request, do_sample=True, temperature=1.0, max_new_tokens=max_new_toks,
                                    return_dict_in_generate=True, output_scores=True, output_logits=True, pad_token_id=pipeline.tokenizer.eos_token_id)

# generated_output = generated_output.sequences[0][-MAX_PARAPHRASE_TOKENS:]
# This works because the prompt is exactly the same for all of them.
# Need to get more picky if the prompt is a different length.
# generated_output = generated_output.sequences[:, batched_paraphrase_request['input_ids'].shape[1]:]
generated_output = generated_output.sequences

elapsed = datetime.datetime.now() - start

print("Total time to do " + str(test_count) + " paraphrases in a batch: " + str(elapsed.total_seconds()))

# print("Batch results: ")
for idx in range(generated_output.shape[0]):
    # Strip eos tokens.
    out_ids = generated_output[idx].cpu().detach().numpy().tolist()
    # Sophistication required for varying prompts... will do that later!
    # out_ids = out_ids[:out_ids.index(pipeline.tokenizer.eos_token_id)]
    out_ids = strip_eot(out_ids, plens[idx], pipeline.tokenizer.eos_token_id)
    # NOTE! Actually a difficult case. Both the prompt and output tokens have padding...
    print(pipeline.tokenizer.decode(out_ids))
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')


print("done")
# It varies a bit, but the batch is indeed faster
# Starting sequential...
# Total time to do 20 paraphrases sequentially: 6.516518
# Starting batch...
# Total time to do 20 paraphrases in a batch: 1.101093

# Even doing 2 is not bad
# Starting sequential...
# Total time to do 2 paraphrases sequentially: 0.577439
# Starting batch...
# Total time to do 2 paraphrases in a batch: 0.372185

# For 200 paraphrases:
# Starting sequential...
# Total time to do 200 paraphrases sequentially: 60.948035
# Starting batch...
# Total time to do 200 paraphrases in a batch: 10.380618
# In this case, it's like a 5x or 6x speedup.

# What about different prompts?

# 1.78 for 20 paras @ 1 each.
# so for 200 qs: 356 seconds total, 4000 qs one at a time.

# so for 200 total qs in a batch: 

# Max batch size probably won't be big... do batch size = number of para attempts.


Start
Starting sequential...


From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Total time to do 30 paraphrases sequentially: 13.806639
Sequential results: 
"'The quick canine cleared the lethargic canine.'<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'A brown fox successfully leapt over a sleeping canine.'"<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'A brown fox successfully cleared a sleeping canine.'<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'The quick reddish-colored animal cleared the sleeping canine.'"<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'A brown fox leaped over a sleeping canine.'"<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'A cunning red creature cleared a distance between itself and a sleeping canine.'"<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'The quick-witted brown animal cleared a considerable height in mid-air.'"<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'A brown fox leaped over a sleepy canine.'<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'The swift fox leaped over the sedentary canine.'<|eot_id|>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
"'A brown fox leaped o

In [ ]:

if False:
    # Debugging get_prompt_gen_logprob
    # Testing very long prompt...
    debug_prompt = ("Repeat the provided sentence exactly, word for word. Don't provide additional explanation besides the sentence.\n"
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "Remember to repeat the sentence."
                    "\nSentence: '{q1}'\n")
    
    # Part of My literal Abstract, for debugging.
    q2 = "With the proliferation of Large Language Model (LLM) chatbots provided via online APIs, " \
    "cheating on school-work has never been so convenient. Developing methods to detecting and prevent academic dishonesty caused by " \
    "LLMs is therefore important. Although much existing work has focused on various methods to detect LLM-generated text, such as through " \
    "the use of watermarking, these approaches are vulnerable to paraphrasing attacks which can seriously degrade detection ability. Instead, " \
    "we present an adversarial approach to “inoculate” short answer questions querying commonly available LLMs to discover questions that are answered."


    the_prob = get_prompt_gen_logprob(pipeline, debug_prompt, q2, q2)
    print("The prob: " + str(the_prob))

    # Should get: 
    # The prob: -0.08437538146972656
    # 12 seconds before fix
    # The prob: -0.08437538146972656
    # Fixed, with no performance hit!

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
torch.cuda.memory_allocated: 11.977356GB
torch.cuda.memory_reserved: 11.990234GB
torch.cuda.max_memory_reserved: 33.244141GB
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
torch.cuda.memory_allocated: 12.202438GB
torch.cuda.memory_reserved: 12.220703GB
torch.cuda.max_memory_reserved: 33.244141GB
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
torch.cuda.memory_allocated: 12.202706GB
torch.cuda.memory_reserved: 12.261719GB
torch.cuda.max_memory_reserved: 33.244141GB
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
torch.cuda.memory_allocated: 12.204499GB
torch.cuda.memory_reserved: 12.435547GB
torch.cuda.max_memory_reserved: 33.244141GB
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
torch.cuda.memory_allocated: 12.207521GB
torch.cuda.memory_reserved: 12.550781GB
torch.cuda.max_memory_reserved: 33.244141GB
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~

In [29]:
# Advanced generation technique: Opposite of Contrastive Decoding
def weighted_prompt_gen(pipeline, promptlist, weightlist, max_tokens, sampling_temp=1.0, perplexity_guide=None):
    # Weigh the logits of multiple prompts - generate output given the weightings
    # Doing this to help improve paraphrase eval odds...

    # get logprob of original question - use as basis score during generation - pass into this function.
    # Generate by weighted addition of multiple prompts -

    weight_tensor = torch.tensor(weightlist)
    
    # Expand the weights tensor for multiplication
    weight_tensor = weight_tensor.unsqueeze(dim=1).expand(-1,pipeline.model.vocab_size)

    weight_tensor = weight_tensor.to(LOCAL_DEVICE_STR)

    encodings_set = []
    for p in promptlist:
        prompt = pipeline.tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        encoded_prompt = pipeline.tokenizer(prompt, return_tensors="pt",
                                             padding=False)  # Keeping padding off for now.
        encodings_set.append(encoded_prompt)

    # No gradients!
    with torch.no_grad():
        # print("Device id: " + str(in_ids.get_device()))

        # Code adapted from here: https://discuss.huggingface.co/t/generate-without-using-the-generate-method/11379
        # TODO: Can transform into batched form? We know all of the needed variations ahead of time...
        # NOTE: I think the model response isn't some starting character, but it could affect the logic here...

        selected_ids = []
        for a in range(max_tokens):

            if len(selected_ids) != 0:
                selected_ids_tensor = torch.Tensor(selected_ids).int().to(LOCAL_DEVICE_STR)
            else:
                selected_ids_tensor = None

            # get logits for every prompt.
            logits_set = []
            for prompt_context in encodings_set:
                in_ids = copy.deepcopy(prompt_context)

                in_ids = in_ids['input_ids'].to(LOCAL_DEVICE_STR)

                # TODO: Fix
                if len(selected_ids) != 0:
                    in_ids = torch.concat([in_ids, torch.unsqueeze(selected_ids_tensor, dim=0)], dim=1)


                outputs = pipeline.model(input_ids=in_ids)
                logits = outputs.logits[:, -1, :] # Final logits
                logits_set.append(logits)

                del in_ids # Clean up memory.

            # Weighted sum
            # So: If a token agrees with more prompts, it will be more likely to be chosen.
            logits_set = torch.concat(logits_set, dim=0)
            # Curious trick: https://discuss.pytorch.org/t/how-to-multiply-each-element-of-a-vector-row-wise-to-matrix/128894/2

            # Force prompts to be weighed equally at every generation step?

            # Interleave strategy: Choose randomly based on weights.
            # np.random.choice(range(len(weightlist), p=))
            # Makes sense in the long run, but probably adds a lot of variation...
            
            # Typical strategy: sum before softmax.
            # combined_logits = torch.sum(torch.mul(logits_set, weight_tensor), dim=0)

            # logit_probs = torch.softmax(combined_logits*sampling_temp,dim=0)

            # Post-sum strategy: Sum only after logits

            combined_logits = torch.softmax(logits_set*sampling_temp,dim=1)

            logit_probs = torch.sum(torch.mul(combined_logits, weight_tensor), dim=0)
            # Negative weights aren't allowed in this combo...

            # Renormalize linearly.
            logit_probs = logit_probs / torch.sum(logit_probs)

            # Decision - sample according to intensity, but how to make decisions based on perplexity_guide?
            # Basically: Weigh the rewrite task heavily early in the generation, but if 
            

            # https://stackoverflow.com/questions/49768306/pytorch-tensor-to-numpy-array
            probs_cpu = logit_probs.detach().cpu().numpy()


            tok_select = np.random.choice(range(logit_probs.shape[0]), p=probs_cpu)
            selected_ids.append(tok_select)

            if selected_ids_tensor is not None:
                del selected_ids_tensor  # Manage tensor memory.

            # is it the end token? Then terminate early
            if selected_ids[-1] == pipeline.tokenizer.eos_token_id:  # eot or eos
                # print("End token token terminate.")
                break

    # Decode according to multi prompts...
    return pipeline.tokenizer.decode(selected_ids)

# Test/debug above function
if False:
    test_prompt_str = ("Repeat the provided sentence exactly, word for word. Don't provide additional explanation besides the sentence."
                    "\nSentence: '{q1}'\n")

    negate_prompt_str = ("Say a sentence that means the opposite of the provided sentence. Don't provide additional explanation besides the sentence."
                       "\nSentence: '{q1}'\n")
    
    q1 = "Hello world!"
    
    promptlist = [test_prompt_str.format(q1=q1), negate_prompt_str.format(q1=q1)]

    max_tokens = 100

    only_pos = weighted_prompt_gen(pipeline, promptlist, [1.0, 0.0], max_tokens, sampling_temp=1.0)
    only_neg = weighted_prompt_gen(pipeline, promptlist, [0.0, 1.0], max_tokens, sampling_temp=1.0)

    print("Positive only: " + str(only_pos))
    print("Negative only: " + str(only_neg))

    # Interesting, the opposite tends to lose...
    combo = weighted_prompt_gen(pipeline, promptlist, [1.0, 1.0], max_tokens, sampling_temp=1.0)
    print("Combo: " + combo)

    # Play with weights
    combo2 = weighted_prompt_gen(pipeline, promptlist, [0.2, 1.0], max_tokens, sampling_temp=1.0)
    print("Combo2: " + combo2)

    # The above debug example is difficult; I'm providing two opposed tasks. How to meaningfully test?
    # Orthogonal tasks?
    gen_task = ("Write a sentence about a fox.") # removed long
    tone_task = ("Write a sentence in a dark tone.") # switching from positive.
    # tone_task = ("Write a question.") # Some tasks do not combo well, or in strange ways.
    promptlist2 = [gen_task, tone_task]

    only_gen = weighted_prompt_gen(pipeline, promptlist2, [1.0, 0.0], max_tokens, sampling_temp=1.0)
    only_tone = weighted_prompt_gen(pipeline, promptlist2, [0.0, 1.0], max_tokens, sampling_temp=1.0)

    comb_tasks = weighted_prompt_gen(pipeline, promptlist2, [1.1, 0.8], max_tokens, sampling_temp=1.0)  # 0.7 for positive tone
    # Note: The weights actually act like a sort of special temperature. Lower for more varied generations, Higher for more consistent gens.
    # Very complicated behavior is observed...

    # comb_neg = weighted_prompt_gen(pipeline, promptlist2, [1.0, -0.3], max_tokens, sampling_temp=1.0)
    print("Gen only: " + only_gen)
    print("Tone only: " + only_tone)
    print("Combo: " + comb_tasks)
    # print("Negative combo: " + comb_neg)

    # Able to demonstrate meaningful combination of prompts if the weights are tuned carefully.

    # Interesting, there is difficulty when switching to the dark tone task.

    # Simpler solution: no eval step????

    # Need separate temp for each prompt?


In [ ]:
# Main
ATTACK_TRIES = 8 # Short for now.
UNIQUENESS_RETRIES = 1 # Short for now.


# Set up openai api stuff.
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

print("Building unmodified question batch...")


# Randomized stochastic result on 50
subsample = 70
eval_df = dev_df.sample(n=subsample)

# eval_df = dev_df

q_orig_set = []
q_only_set = []

# Build batch
for q_idx in range(eval_df.shape[0]):
    orig_question_str = format_para_question_single(eval_df.iloc[q_idx, 0], eval_df, q_idx, include_answer=False)

    q_orig_set.append(orig_question_str)
    q_only_set.append(eval_df.iloc[q_idx, 0])
    
    # print("Batching Question: " + str(q_idx+1) + "/" + str(eval_df.shape[0]))

# 98 seconds to paraphrase & query 6 correct q's 
# estimate: 1 hour to evaluate the whole dev set... let's go...

# q_orig_set = q_orig_set[:10]  # 78 seconds for 5; estimating 74 mins for the whole dev set at least...
# q_only_set = q_only_set[:10]

mdl_requests = build_question_batch(OPENAI_TEST_MODEL, q_orig_set, max_new_tokens=RESPONSE_MAX_NEW_TOKENS)

print("Submitting question batch...")
responses = bulk_eval(mdl_requests, openai_client)
print("Batch responses retrieved.")

q_correct_idxs = []

count_noncompliant = 0

# Get 
for r_idx, answer in enumerate(responses):
    label = eval_df.iloc[r_idx, eval_df.shape[1]-1]  # Last column.
    
    # TODO: Stats on noncompliant responses?

    if re.search(ANSWER_REG, answer) is not None:
        # Does the answer match? Yes or no...
        if ANSWER_PREFIX.format(label) in answer:
            q_correct_idxs.append(r_idx)
    else:
        print("Answer is noncompliant: " + answer)
        print("Answer length: " + str(len(answer)))
        count_noncompliant += 1

num_initial_correct = len(q_correct_idxs)
initial_accuracy = num_initial_correct / len(q_orig_set)
print("Total question count: " + str(len(q_orig_set)))
print("Accuracy: " + str(initial_accuracy))

print("Count noncompliant: " + str(count_noncompliant))

# Find incorrectly answered questions. Paraphrase these.
print("Paraphrasing correct questions; Question count: " + str(num_initial_correct))

print("Paraphrases per question: " + str(ATTACK_TRIES))

# So batch request length is ATTACK_TRIES * len(q_incorrect_idxs)

# para_temp = 0.7  # Lower temp?
# Actually works: Definitionally, lower temp makes it easy to pass the paraphrase evaluation step...
# But paraphrase quality suffers...
para_temp = 1.0

max_para_retries = 2  # can't afford longer right now.

num_successful = 0

num_paras_mapping = []  # remember how many paraphrases were creatd.
paras_batch_set = []
for progress_idx, r_index in enumerate(q_correct_idxs):
    print("Paraphrasing: " + str(progress_idx+1) + "/" + str(num_initial_correct))
    question = eval_df.iloc[r_index, 0]

    para_set = []
    for a in range(ATTACK_TRIES + UNIQUENESS_RETRIES):
        # Use llama to paraphrase

        # NOTE: Exclude duplicate paraphrases...
        # JL NOTE: Using simple reject/retry technique.

        para_q, tries = paraphrase_by_perplexity(pipeline, question, max_retries=max_para_retries, temp=para_temp, max_new_toks=MAX_PARAPHRASE_TOKENS)

        if para_q is not None:

            # check if equal to anything.
            for exist_p in para_set:
                if para_q == exist_p:
                    break

            para_set.append(para_q)
            if len(para_set) == ATTACK_TRIES:
                # num_paras_mapping == ATTACK_TRIES
                break

    # TODO: Handle paraphrase failure?
    num_paras_mapping.append(len(para_set))
    if(len(para_set) != 0):
        num_successful += 1

    # Format questions.
    paras_batch_set = paras_batch_set + [format_para_question_single(x, eval_df, r_index, include_answer=False) for x in para_set]

print("Proportion of questions with successful paraphrase: " + str(num_successful/len(q_correct_idxs)))

para_mdl_requests = build_question_batch(OPENAI_TEST_MODEL, paras_batch_set, max_new_tokens=RESPONSE_MAX_NEW_TOKENS)

print("Submitting question batch...")
para_responses = bulk_eval(para_mdl_requests, openai_client)
print("Batch responses retrieved.")

with open("/home/jlim/MS_Project/Paraphrase_Attack/summary_data.pickle", 'wb') as f:
    # Save some data for record keeping.
    summary_data = {
        'initial_responses': responses,
        'correct_idxs': q_correct_idxs,
        'num_paras_mapping': num_paras_mapping,
        'para_responses': para_responses,
    }
    pickle.dump(summary_data, f)

# Parse using num_paras_mapping as well

# Calculate ASR.
# Save responses, paraphrased Qs?

# Just try ASR first...

count_incorrect = 0

for idx, mapping in enumerate(num_paras_mapping):
    base_idx = 0
    for j in range(0, idx):
        base_idx += num_paras_mapping[j]

    for subidx in range(mapping):
        # Actually need to do cumulative sum
        full_idx = base_idx + subidx
        # Answer correct or not?

        # label = eval_df.iloc[full_idx, eval_df.shape[1]-1]  # Last column.

        # Reindex into original labels.
        label = eval_df.iloc[q_correct_idxs[idx], eval_df.shape[1]-1]  # Last column.
    
        # TODO: Stats on noncompliant responses?
        answer = para_responses[full_idx]



        if re.search(ANSWER_REG, answer) is not None:
            # Does the answer match? Yes or no...
            if not (ANSWER_PREFIX.format(label) in answer):
                count_incorrect += 1

                # Print only successful attacks.
                print()
                print("Original Q: " + eval_df.iloc[q_correct_idxs[idx], 0])

                # Question
                print("Paraphrased Q: " + paras_batch_set[full_idx])

                # Model answer
                print("Model answer: " + answer)

                # Correct answer
                print("Correct answer: " + label)

                print("Model output incorrect (Attack Success?)")
                break  # Attack successful, skip to next paraphrase set.
            # else:
            #     print("Still correct instance (sanity check!)")
        else:
            print("Answer is noncompliant: " + answer)

        # # Does the answer match? Yes or no...
        # if not (ANSWER_PREFIX.format(label) in answer):
        #     count_incorrect += 1
        #     break  # Attack successful, skip to next paraphrase set.
        # else:
        #     print("Still correct instance (sanity check!)")

asr = count_incorrect / num_initial_correct
print("Attack Success Proportion: " + str(asr))

# is_proportion = (len(para_q) > len(q1) - char_len_proportion*len(q1)) and (len(para_q) < len(q1) + char_len_proportion*len(q1))

Building unmodified question batch...
Submitting question batch...
Completion: 1/70
Completion: 2/70
Completion: 3/70
Completion: 4/70
Completion: 5/70
Completion: 6/70
Completion: 7/70
Completion: 8/70
Completion: 9/70
Completion: 10/70
Completion: 11/70
Completion: 12/70
Completion: 13/70
Completion: 14/70
Completion: 15/70
Completion: 16/70
Completion: 17/70
Completion: 18/70
Completion: 19/70
Completion: 20/70
Completion: 21/70
Completion: 22/70
Completion: 23/70
Completion: 24/70
Completion: 25/70
Completion: 26/70
Completion: 27/70
Completion: 28/70
Completion: 29/70
Completion: 30/70
Completion: 31/70
Completion: 32/70
Completion: 33/70
Completion: 34/70
Completion: 35/70
Completion: 36/70
Completion: 37/70
Completion: 38/70
Completion: 39/70
Completion: 40/70
Completion: 41/70
Completion: 42/70
Completion: 43/70
Completion: 44/70
Completion: 45/70
Completion: 46/70
Completion: 47/70
Completion: 48/70
Completion: 49/70
Completion: 50/70
Completion: 51/70
Completion: 52/70
Comple

: 

What is the most probable cause of a 44-year-old man's 3-day history of sore throat, cough, runny nose, morning headache, and limited relief with ibuprofen, in the presence of nonspecific findings on physical examination?
What is the most probable cause of a 44-year-old man's 3-day history of sore throat, cough, runny nose, morning headache, and limited relief with ibuprofen, in the presence of nonspecific findings on physical examination?

Notes:

-Using gpt4o-mini
-Takes a very long time at 1000 tokens per... (like 15 mins for the set) will shorten to 240?

-For gpt4o-mini, 50 tokens generated (5 min eval)
(For 285 questions)
Accuracy: 0.3929824561403509
Count noncompliant: 159
Paraphrasing correct questions; Question count: 112

-no COT, change prompt to output answer first.
Accuracy: 0.7052631578947368
Count noncompliant: 25
Paraphrasing correct questions; Question count: 201
Paraphrases per question: 5

-Run #2:
Answer length: 127
Total question count: 285
Accuracy: 0.7263157894736842
Count noncompliant: 20
Paraphrasing correct questions; Question count: 207
Paraphrases per question: 5

-Improve compliance.
--This also improved accuracy too.
--Runtime for first eval is like 2-3 mins.
Total question count: 285
Accuracy: 0.7508771929824561
Count noncompliant: 3
Paraphrasing correct questions; Question count: 214

-Run #2:
-- Compliance is good.
Total question count: 285
Accuracy: 0.7578947368421053
Count noncompliant: 1
Paraphrasing correct questions; Question count: 216

-ISSUE: Difficult to get a compliant paraphrase - lots of regeneration/retries needed...

-Trying hail mary approach, seeing results on full dev set...
-Promising? But could be misleading...

-Will start recording questions/categories that cause issues:
-
-Was like 53 mins for 50 Qs. Not good...

-Got some weak results, but it took 33 minutes...
Total question count: 50
Accuracy: 0.68
Count noncompliant: 1
Paraphrasing correct questions; Question count: 34

Submitting question batch...
Completion: 1/91

Proportion of questions with successful paraphrase: 0.8235294117647058
Attack Success Proportion: 0.23529411764705882

-Attempting longer run with more attacks. I'm assuming like 1.5 hours...

